# Norm Intermezzo: Context of magnitude/length/norm of a vector explained.

The norm (or magnitude or length) of a vector describe the same thing: the distance between the origin ('0') and the coordinate of the vector.


- What properties does the norm of a vector describe?
- What pattern is there to be found in number of words and its vector length?

### Settings

Settings to specify which experiment will be analyzed

In [ ]:
# File paths
experiment_name = "free_1000_251013_pest_PD"

embedding_file = f"embedding_{experiment_name}.pickle"
raw_corpus_file = f"corpus_{experiment_name}.csv"

## Initialization

### Imports

In [ ]:
import numpy as np
import pandas as pd
import pickle
from statsmodels.stats.diagnostic import normal_ad

import matplotlib.pyplot as plt

### Functions

In [ ]:
# function of description above
def get_norm(matrix):
    """Calculate the norm of a vector"""
    norm = np.sqrt(
        np.sum(
            np.power(
                matrix, 
                2
            ),
        axis=1
        )
    )

    return norm

# function to divide vector by norm
# (..probably there is a numpy function for it.. but i failed to find it)
def normalize(matrix, scales):
    """Divide elements in matrix by given scale"""
    return np.array([vector / scales[idx] for idx, vector in enumerate(matrix)])

In [ ]:
def to_series(M):
    return pd.Series(M)

In [ ]:
def get_n_chars(df, idx):
    return df.loc[idx, "sentence_text"].str.len()

def get_n_words(df, idx):
    return df.loc[idx, "sentence_text"].str.split(' ').map(len)

In [ ]:
def DS_Q_Q_Plot(y, est = 'robust', title = 'Q-Q plot', **kwargs):
    """
    *
    Function DS_Q_Q_Plot(y, est = 'robust', **kwargs)
    
       This function makes a normal quantile-quantile plot (Q-Q-plot), also known
       as a probability plot, to visually check whether data follow a normal distribution.
    
    Requires:            numpy, scipy.stats.iqr, scipy.stats.norm, matplotlib.pyplot
    
    Arguments:
      y                  data array
      est                Estimation method for normal parameters mu and sigma:
                         either 'robust' (default), or 'ML' (Maximum Likelihood),
                         or 'preset' (given values)
      N.B. If est='preset' than the *optional* parameters mu, sigma must be provided:
      mu                 preset value of mu
      sigma              preset value of sigma
      title              Title for the graph (default: 'Q-Q plot')
      
    Returns:
      Estimated mu, sigma, n, and expected number of datapoints outside CI in Q-Q-plot.
      Q-Q-plot
      
    Author:            M.E.F. Apol
    Date:              2020-01-06, revision 2022-08-30, 2023-12-19
    """
    
    import numpy as np
    from scipy.stats import iqr # iqr is the Interquartile Range function
    import matplotlib.pyplot as plt
    from scipy.stats import norm
    
    # First, get the optional arguments mu and sigma:
    mu_0 = kwargs.get('mu', None)
    sigma_0 = kwargs.get('sigma', None)
    
    n = len(y)
    
    # Calculate order statistic:
    y_os = np.sort(y)
  
    # Estimates of mu and sigma:
    # ML estimates:
    mu_ML = np.mean(y)
    sigma2_ML = np.var(y)
    sigma_ML = np.std(y) # biased estimate
    s2 = np.var(y, ddof=1)
    s = np.std(y, ddof=1) # unbiased estimate
    # Robust estimates:
    mu_R = np.median(y)
    sigma_R = iqr(y)/1.349

    # Assign values of mu and sigma for z-transform:
    if est == 'ML':
        mu, sigma = mu_ML, s
    elif est == 'robust':
        mu, sigma = mu_R, sigma_R
    elif est == 'preset':
        mu, sigma = mu_0, sigma_0
    else:
        print('Wrong estimation method chosen!')
        return()
        
    print('Estimation method: ' + est)
    print('n = {:d}, mu = {:.4g}, sigma = {:.4g}'.format(n, mu,sigma))
    
    # Expected number of deviations (95% confidence level):
    n_dev = np.round(0.05*n)
    
    print('Expected number of data outside CI: {:.0f}'.format(n_dev))
         
    # Perform z-transform: sample quantiles z.i
    z_i = (y_os - mu)/sigma

    # Calculate cumulative probabilities p.i:
    i = np.array(range(n)) + 1
    p_i = (i - 0.5)/n

    # Calculate theoretical quantiles z.(i):
    z_th = norm.ppf(p_i, 0, 1)

    # Calculate SE or theoretical quantiles:
    SE_z_th = (1/norm.pdf(z_th, 0, 1)) * np.sqrt((p_i * (1 - p_i)) / n)

    # Calculate 95% CI of diagonal line:
    CI_upper = z_th + 1.96 * SE_z_th
    CI_lower = z_th - 1.96 * SE_z_th

    # Make Q-Q plot:
    plt.plot(z_th, z_i, 'o', color='k', label='experimental data')
    plt.plot(z_th, z_th, '--', color='r', label='normal line')
    plt.plot(z_th, CI_upper, '--', color='b', label='95% CI')
    plt.plot(z_th, CI_lower, '--', color='b')
    plt.xlabel('Theoretical quantiles, $z_{(i)}$')
    plt.ylabel('Sample quantiles, $z_i$')
    plt.title(title + ' (' + est + ')')
    plt.legend(loc='best')
    plt.show()
    pass;

In [ ]:
def DS_Q_Q_Hist(y, est='robust', title = 'Histogram with corresponding normal distribution', 
                xlabel = 'Values, $y$', ylabel = 'Probability $f(y)$', bins='auto', align='mid', **kwargs):
    """
    *
    Function DS_Q_Q_Hist(y, est='robust', **kwargs)
    
       This function makes a histogram of the data and superimposes a fitted normal
       distribution.
       
    Requires:            - 
    
    Arguments:
      y                  data array
      est                Estimation method for normal parameters mu and sigma:
                         either 'robust' (default), or 'ML' (Maximum Likelihood),
                         or 'MM' (Method of Moments) or 'preset' (given values)
      N.B. If est='preset' than the optional parameters mu, sigma MUST be provided:
      mu                 preset value of mu
      sigma              preset value of sigma
      title              Title of the graph (default: 'Histogram with corresponding normal distribution')
      xlabel             x-label of the graph (default: 'Values, $y$')
      ylabel             y-label of the graph (default: 'Probability $f(y)$')
      bins               bins used for histogram (default: 'auto'); can be np.array of bins
      align              align the histogram bins (default: 'mid'); for already binned data the option 'left' might
                         give better results
    
    Returns:
      Estimations of mu and sigma
      Histogram of data with estimated normal distribution superimposed
      
    Author:            M.E.F. Apol
    Date:              2020-01-06, adaptions 2023-12-19
    """
    
    import numpy as np
    from scipy.stats import iqr # iqr is the Interquartile Range function
    from scipy.stats import norm
    import matplotlib.pyplot as plt
    
    # First, get the optional arguments mu and sigma:
    mu_0 = kwargs.get('mu', None)
    sigma_0 = kwargs.get('sigma', None)
    
    n = len(y)
    
    # Estimates of mu and sigma:
    # ML estimates:
    mu_ML = np.mean(y)
    sigma2_ML = np.var(y) # biased estimate
    sigma_ML = np.std(y) 
    s2 = np.var(y, ddof=1) # unbiased estimate
    s = np.std(y, ddof=1) 
    # Robust estimates:
    mu_R = np.median(y)
    sigma_R = iqr(y)/1.349

    # Assign values of mu and sigma for z-transform:
    if est == 'ML':
        mu, sigma = mu_ML, s    
    elif est == 'MM':
        mu, sigma = mu_ML, sigma_ML
    elif est == 'robust':
        mu, sigma = mu_R, sigma_R
    elif est == 'preset':
        mu, sigma = mu_0, sigma_0
    else:
        print('Wrong estimation method chosen!')
        return()
    print('Estimation method: ' + est)
    print('mu = {:.4g}, sigma = {:.4g}'.format(mu,sigma))
        
    # Calculate the CLT normal distribution:
    if bins != 'auto':
        y_min = np.min([np.min(y), np.min(bins)])
        y_max = np.max([np.max(y), np.max(bins)])
    else:
        y_min = np.min(y)
        y_max = np.max(y)
    x = np.linspace(y_min, y_max, 501)
    rv = np.array([norm.pdf(xi, loc = mu, scale = sigma) for xi in x])
    
    # Make a histogram with corresponding normal distribution:
    plt.hist(x=y, density=True, bins=bins, align=align,
             color='darkgrey',alpha=1, rwidth=1, label='experimental')
    plt.plot(x, rv, 'r', label='normal approximation')
    plt.grid(axis='y', alpha=0.5)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title + ' (' + est + ')')
    plt.legend(loc='best')
    plt.show()
    pass;

In [ ]:
def DS_AndersonDarling_test_normal(y, alpha=0.05):
    """
    *
    Function DS_AndersonDarling_test_normal(y, alpha)
    
       This function tests whether the data y follow a normal distribution (Null Hypothesis Significance Test).
    
    Requires:            scipy.stats.anderson
    
    References:          * Th. Anderson & D. Darling (1952) - "Asymptotic Theory of 
                         Certain "Goodness of Fit" Criteria Based on Stochastic Processes".
                         Ann. Math. Statist. 23, 193-212. DOI: 10.1214/aoms/1177729437
                         * R.B. D'Agostino (1986). "Tests for the Normal Distribution"
                         In: R.B. D'Augostino & M.A. Stephens - "Goodness-of-fit
                         techniques", Marcel Dekker.
    
    Arguments:
      y                  data array
      alpha              significance level of the critical value (default: alpha = 0.05)
      
    Usage:               DS_AndersonDarling_test_normal(y, alpha=alpha)
      
      
    Returns:             AD, AD_star, p_value [ + print interpretable output to stdout ]
    where
      AD                 (Large-sample) Anderson-Darling statistic
      AD_star            Small-sample Anderson-Darling statistic
      p_value            p-value of AD-test
      
    Author:            M.E.F. Apol
    Date:              2023-12-05
    """
    
    from scipy.stats import anderson
    
    AD = anderson(y, dist='norm').statistic
    n = len(y)
    AD_star = AD*(1 + 0.75/n + 2.25/n**2)
    
    # p-values based on D'Augostino & Stephens (1986):
    if(AD_star <= 0.2): # Eq. (1)
        p_value = 1 - np.exp(-13.436 + 101.14*AD_star - 223.73*AD_star**2)
    elif((AD_star > 0.2) & (AD_star <= 0.34)): # Eq. (2)
        p_value = 1 - np.exp(-8.318 + 42.796*AD_star - 59.938*AD_star**2)
    elif((AD_star > 0.34) & (AD_star < 0.6)): # Eq. (3)
        p_value = np.exp(0.9177 - 4.279*AD_star - 1.38*AD_star**2)
    elif(AD_star >= 0.6): # Eq. (4)
        p_value = np.exp(1.2937 - 5.709*AD_star + 0.0186*AD_star**2)
        
    # Critical AD* values, based on D'Augostino & Stephens (1986):
    # Inverting these relations, we get
    # Invert (1) if alpha > 0.884
    # Invert (2) if 0.50 < alpha < 0.884
    # Invert (3) if 0.1182 < alpha < 0.50
    # Invert (4) if alpha < 0.1182
    
    if(alpha >= 0.884): # Eq. (1a)
        AD_crit = (-101.14+np.sqrt(101.14**2-4*-223.73*(-13.436-np.log(1-alpha))))/(2* -223.73)
    elif((alpha < 0.884) & (alpha >= 0.50)): # Eq. (2a)
        AD_crit = (-42.796+np.sqrt(42.796**2-4* -59.938*(-8.318-np.log(1-alpha))))/(2* -59.938)
    elif((alpha < 0.50) & (alpha >= 0.1182)): # Eq. (3a)
        AD_crit = (4.279-np.sqrt(4.279**2-4* -1.38*(0.9177-np.log(alpha))))/(2* -1.38)
    elif(alpha < 0.1182): # Eq. (4a)
        AD_crit = (5.709-np.sqrt(5.709**2-4*0.0186*(1.2937-np.log(alpha))))/(2*0.0186)
    
    # Additional statistics:
    y_av = np.mean(y)
    s = np.std(y, ddof=1)
    
    print(80*'-')
    print('Anderson-Darling-test for normality of data:')
    print('     assuming Normal(mu | sigma2) data for dataset')
    print('y.av = {:.3g}, s = {:.3g}, n = {:d}, alpha = {:.3g}'.format(y_av, s, n, alpha))
    print('H0: data follows normal distribution')
    print('H1: data does not follow normal distribution')
    print('AD = {:.3g}, AD* = {:.3g}, p-value = {:.3g}, AD*.crit = {:.3g}'.format(AD, AD_star, p_value, AD_crit))
    print(80*'-')
    
    return(AD, AD_star, p_value);

In [ ]:
def DS_2sample_MannWhitney_test_medians(y1, y2, alternative='two-sided', alpha=0.05):
    """
    *
    Function DS_2sample_MannWhitney_test_medians(y1, y2, alternative='two-sided', alpha=0.05)
    
       This function tests two medians eta.1 and eta.2 of data y1 and y2 (Null Hypothesis Significance Test).
       The distributions are assumed to be identical, but not necessarily normal.
    
    Requires:            scipy.stats.mannwhitneyu
    
    Usage:               DS_2sample_MannWhitney_test_medians(y1, y2, 
                              alternative=['two-sided']/'less'/'greater', alpha=0.05)
    
    Arguments:
      y1, y2             data arrays
      alternative        'two-sided' [default]   H1: eta.1 != eta.2
                         'less'                  H1: eta.1 <  eta.2
                         'greater'               H1: eta.1 >  eta.2
      alpha              significance level of test [default: 0.05]     
 
 
    Returns:             U.1, U.2, U, p_value [ + print interpretable output to stdout ]
    where
      U.1, U.2, U        Mann-Whitney statistics
      p_value            p-value of Mann-Whitney U-test
      
    Validation:          against SPSS v. 28
      
    Author:            M.E.F. Apol
    Date:              2023-12-20, revision 2024_01_02
    """

    from scipy.stats import mannwhitneyu
    
    # Additional statistics:
    y_med_1 = np.median(y1)
    y_med_2 = np.median(y2)
    n_1 = len(y1)
    n_2 = len(y2)
    
    print(80*'-')
    print('2-sample Mann-Whitney U-test for 2 medians:')
    print('     assuming identical distributions for both datasets, that may differ in location')
    print('y.med.1 = {:.4g}, y.med.2 = {:.4g}, n.1 = {:d}, n.2 = {:d}, alpha = {:.3g}'.format(y_med_1, y_med_2, n_1, n_2, alpha))
    print('H0: eta.1  = eta.2')
    
    if alternative == 'two-sided':
        print('H1: eta != eta*')
    elif alternative == 'greater':
        print('H1: eta  > eta*')
    elif alternative == 'less':
        print('H1: eta  < eta*')
    else:
        print('Wrong alternative hypothesis chosen!')
        print(80*'-' + '\n')
        U_1, U_2, U, p_value = np.nan, np.nan, np.nan, np.nan
        return(U_1, U_2, U, p_value)
    
    #res = mannwhitneyu(y1, y2, alternative=alternative, use_continuity=True)
    res = mannwhitneyu(y1, y2, alternative=alternative, use_continuity=False)
    U_1 = res.statistic
    U_2 = n_1*n_2 - U_1
    U = np.min([U_1, U_2])
    p_value = res.pvalue
    
    # -- TO DO -- Find formula for U.crit
    U_crit = np.nan
    
    # To compute an effect size for the signed-rank test, one can use the rank-biserial correlation?
    
    # Correlation coefficient:
    # From: datatab.net (dd 2023_12_19), https://maths.shu.ac.uk/mathshelp/ (dd 2023_12_19)
    # Normal approximation:
    # Expectation value:
    mu_U = n_1*n_2/2
    # Standard deviation:
    sigma_U = np.sqrt(n_1*n_2*(n_1+n_2+1)/12)
    # Standard normal statistic:
    z = (U - mu_U) / sigma_U
    # Effect size:
    r = z / np.sqrt(n_1+n_2)
    
    # Point biserial correlation r.pb
    # From: https://www.andrews.edu/~calkins/math/edrm611/edrm13.htm
    # Additional statistics:
    y_av_1 = np.mean(y1)
    y_av_2 = np.mean(y2)
    s2_1 = np.var(y1, ddof=1)
    s2_2 = np.var(y2, ddof=1)
    s2_p = ((n_1-1)*s2_1 + (n_2-1)*s2_2)/(n_1+n_2-2)
    s_p = np.sqrt(s2_p)
    r_pb = (y_av_1 - y_av_2)/s_p * np.sqrt(n_1*n_2)/(n_1 + n_2)
    
    print('U.1 = {:.3g}, U.2 = {:.3g}, U = {:.3g}, p-value = {:.3g}, z = {:.3g}, U.crit = {:.3g}'.format(U_1, U_2, U, p_value, z, U_crit))
    print(80*'.')
    print('Effect size: r    = {:.3g}; benchmarks |r|: 0.1 = small, 0.3 = medium, 0.5 = large'.format(r))
    # print('Effect size: r.pb = {:.3g}; benchmarks r: 0.1 = small, 0.3 = medium, 0.5 = large (?)'.format(r_pb))
    print(80*'-')
    
    return(U_1, U_2, U, p_value);

### Load files

In [ ]:
# Load raw corpus
df_corpus = pd.read_csv(f'../../../data/corpus/{raw_corpus_file}')

# Load sentence embeddings
with open(f"../../../data/vectors/{embedding_file}", 'rb') as handle:
    embeddings = pickle.load(handle)

## What properties does the norm of a sentence vector describe?

For word vectors, Depending on the embedding method that was used, it can represent its uniqueness in meaning (co-occurrence matrix; like the word 'star' can have different meanings, but jargon often has a single meaning) or the frequency to which a word is present (bag of words model).
This made me wonder if the norm of a sentence vector can describe a certain property.

First we need to check if the embedding was already normalized by the embedder. (This should not be the case if we want to continue.)

In [ ]:
# Check if embeddings normalized
if np.sum(get_norm(embeddings)) == embeddings.shape[0]:
    print("Embedding is already normalized. (`normalized_embeddings == embeddings`)")
    normalized_embeddings = embeddings

else:
    print("Embedding is not normalized.")

In [ ]:
# Get the norm of the vectors
M_norm = get_norm(embeddings)

### Plot distribution

Plot distribution of vector norms and show sentences that are in the upper and lower part of the distribution.

In [ ]:
# Plot distribution of vector norms
plt.hist(
    M_norm,
    bins= np.arange(10,16,0.25)
)

plt.title("Distribution of sentence vector norms")
plt.xlabel("Vector norm")
plt.ylabel("Frequency")

plt.show()

In [ ]:
DS_Q_Q_Hist(M_norm)

In [ ]:
DS_Q_Q_Plot(M_norm)

In [ ]:
# Describe
describe = pd.Series(M_norm).describe()
describe

In [ ]:
print("p-value (if p-value < 0.05, then data is not normally distributed.)")
p_value = normal_ad(M_norm)[1]
string = f"var is NOT normal distributed (p-value = {p_value})" if p_value < 0.05 else f"var IS normal distributed (p-value = {p_value})"
print(string)

The norms of the sentence vectors appear to be normal distributed, but it is not.

### Sample sentence vectors with small, mid, and large norms

In [ ]:
# Get indexes of vectors where norm is:
# larger than 15
idx_high_norm = np.where(M_norm > describe['mean'] + describe['std'] *3)[0][:25]

# smaller than 11.25
idx_low_norm = np.where(M_norm < describe['mean'] - describe['std'] *3)[0][:25]

# near mean (and only take 24 sentences)
idx_mean_norm = np.where((describe['mean'] - 0.1 < M_norm) & (M_norm < describe['mean'] + 0.1))[0][:25]

print(f"idx_high_norm number of sentences: {idx_high_norm.shape[0]}")
print(f"idx_mean_norm number of sentences: {idx_mean_norm.shape[0]}")
print(f"idx_low_norm number of sentences: {idx_low_norm.shape[0]}")

#### Large norm

Summary: Vectors with a large norm in our corpus are short sentences (3 words long on average, where the largest is 7 words long). (Is this what we expect?)

Current 'large' sub selection is the `mean + 3* std`:

In [ ]:
# Describe vectors
print("Describe selected long vectors")
L_d = to_series(M_norm[idx_high_norm])
L_d.describe()

Next we will print the sentences:

In [ ]:
print("\n~~~\n".join(df_corpus.loc[idx_high_norm, "sentence_text"].values))

Sentences appear to be rather short. With sometimes the same sentence (which would make sense, since they would share the same vector coordinates.)

In [ ]:
# Described number of characters per sentence
print("Described characters")
L_chars = get_n_chars(df_corpus, idx_high_norm)
L_chars.describe()

In [ ]:
# Described number of words per sentence
print("Described words")
L_words = get_n_words(df_corpus, idx_high_norm)
L_words.describe()

'Described number of words per sentence' confirms that sentences appear short. Let's compare with the other norms:

#### Average norm

Summary: A lot of variety in sentence meaning. Most of them are suspected to be from the 'Material and Method' / 'Results' section.
Average character and word count appear to be significant different than the 'large' norm sentence vectors.

Current 'average' sub selection is the `mean - 0.1`:

In [ ]:
# Describe vectors
print("Describe selected average vectors")
A_d = to_series(M_norm[idx_mean_norm])
A_d.describe()

Next we will print the sentences:

In [ ]:
print("\n~~~\n".join(df_corpus.loc[idx_mean_norm, "sentence_text"].values))

There appears to be a large variety of meaning. We suspect a lot of sentences come from the 'materials and methods' and 'results' section.

In [ ]:
# Described number of characters per sentence
print("Described characters")
A_chars = get_n_chars(df_corpus, idx_mean_norm)
A_chars.describe()

In [ ]:
# Described number of words per sentence
print("Described words")
A_words = get_n_words(df_corpus, idx_mean_norm)
A_words.describe()

Average character and word count appear to be significant different than the 'large' norm sentence vectors.

#### Small norm

Current 'small' sub selection is the `mean - 3* std`:

In [ ]:
# Describe vectors
print("Describe selected short vectors")
S_d = to_series(M_norm[idx_low_norm])
S_d.describe()

Next we will print the sentences:

In [ ]:
print("\n~~~\n".join(df_corpus.loc[idx_low_norm, "sentence_text"].values))

Again a large variety of meaning. And three sentences that start with 'To our knowledge'.

In [ ]:
# Described number of characters per sentence
print("Described characters")
S_chars = get_n_chars(df_corpus,idx_low_norm)
S_chars.describe()

In [ ]:
# Described number of words per sentence
print("Described words")
S_words = get_n_words(df_corpus,idx_low_norm)
S_words.describe()

The absolute average number of words is larger of the 'small' norm vectors, compared to the 'average' norm vectors.
Let's see if it is significant in the next section.

### Summary of described sentences

Comparing the found properties from previous section:

In [ ]:
plt.boxplot(
    [L_d, A_d, S_d]
)

plt.title("Vector norms of subsets")
plt.xticks([1, 2, 3], ['Large', 'Average', 'Small'])
plt.ylabel("Norm of vectors")

plt.show()

Again, overview of difference in the subsets.

In [ ]:
plt.boxplot(
    [L_chars, A_chars, S_chars]
)

plt.title("Number of characters")
plt.xticks([1, 2, 3], ['Large', 'Average', 'Small'])
plt.ylabel("Character frequency")

plt.show()

Then test if larger norm subset has shorter sentences than the average norm subset.

In [ ]:
# Overall distribution
O_words = df_corpus['sentence_text'].str.split(' ').map(len)
plt.hlines([O_words.mean()], 
           xmin= 0, xmax=4, 
           color='grey', alpha=0.5, label="Overall mean")
plt.axhspan(
    ymax = O_words.mean() + O_words.std()*2,
    ymin = O_words.mean() - O_words.std()*2,
    color = 'grey',
    alpha= 0.1,
    label= "Overall 95%"
)
plt.axhspan(
    ymax = O_words.mean() + O_words.std(),
    ymin = O_words.mean() - O_words.std(),
    color = 'grey',
    alpha= 0.1,
    label= "Overall 68%"
)

# Plot boxplot subsets
plt.boxplot(
    [L_words, A_words, S_words]
)


plt.title("Number of words")
plt.legend()
plt.xticks([1, 2, 3], ['Large', 'Average', 'Small'])
plt.ylabel("Word frequency")

plt.show()

##### Does the large norm subset have significant less words than the average subset?

Yes. Even the 'average' subset has less words than 'small' norms:

In [ ]:
# Large < Average?
DS_2sample_MannWhitney_test_medians(
    L_words.to_numpy(), 
    A_words.to_numpy(), 
    alternative='less'
    )

In [ ]:
# Average < Small?
DS_2sample_MannWhitney_test_medians(
    A_words.to_numpy(), 
    S_words.to_numpy(), 
    alternative='less'
    )

## What pattern is there to be found in number of words and its vector length?

To the corpus we add the columns 'n_words' for the number of words in the sentence, and 'vector_norm' to indicate length of the norm.

If there is a correlation between vector_norm and number of words, we should see a straight line when plotting `the number of words` against `the vector_norm`.

In [ ]:
# Add column with number of words
df_corpus["n_words"] = df_corpus["sentence_text"].str.split(' ').map(len)

In [ ]:
# Add column with number of words
df_corpus["n_char"] = df_corpus["sentence_text"].str.len()

In [ ]:
# Add column with vector norm
df_corpus['vector_norm'] = M_norm

In [ ]:
# Select these two columns
selection = df_corpus[['n_words', 'n_char','vector_norm']]
selection

In [ ]:
# Plot these two columns
plt.scatter(
    # selection['vector_norm'],
    selection["n_words"],
    # selection["n_char"],
    selection['vector_norm'],
    alpha=0.05
)

plt.xlabel("Word count")
plt.ylabel("Vector norm")
plt.title("Vector norm vs word count per sentence")

plt.show()

There does not appear to be a strong correlation. 
The subset with large vector norms have a less words in their sentence, but a norm vector around the mean (13) have a wide range of word counts per sentence. This range appear to get smaller the smaller the vector norm becomes again, but with the average word count still higher than the large vector norms.

So I'll leave this exploration for now.